# YOLOv8s GCBlock 3-Class on Kaggle

这份 notebook 用于在 Kaggle 上训练 `Baseline + Neck-only GCBlock`，目标类别只有 3 个：

- `0: chuck`
- `1: gripper`
- `2: drill_pipe`

设计原则：

- 只在 neck 中加入 `GCBlock`
- `Detect head` 保持原始 `Detect`
- 使用 `yolov8s.pt`
- 使用 `SGD`
- 不启用 `QFL / ATFL / TSCODE / STC / sampler / 自定义 loss`
- 基于 GitHub 仓库里的自定义 `GCBlock` 代码在 Kaggle 长训


In [ ]:
!nvidia-smi


In [ ]:
# 固定数据集路径，优先用你现在的 Kaggle 私有数据集路径
from pathlib import Path

CANDIDATE_DATASET_ROOTS = [
    Path('/kaggle/input/yolo-datasets-10k-yolov8-3class'),
    Path('/kaggle/input/datasets/yuanfangshang/yolo-datasets-10k-yolov8-3class'),
]

DATASET_ROOT = next((p for p in CANDIDATE_DATASET_ROOTS if p.exists()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError(f'Cannot find dataset root in: {CANDIDATE_DATASET_ROOTS}')

print('DATASET_ROOT =', DATASET_ROOT)
print('images/train exists =', (DATASET_ROOT / 'images' / 'train').exists())
print('labels/train exists =', (DATASET_ROOT / 'labels' / 'train').exists())


In [ ]:
# 生成 Kaggle 版 3 类 data.yaml，直接指向 /kaggle/input
from pathlib import Path

yaml_text = f'''path: {DATASET_ROOT.as_posix()}
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: chuck
  1: gripper
  2: drill_pipe
'''

DATA_CFG = Path('/kaggle/working/coalmine3_kaggle.yaml')
DATA_CFG.write_text(yaml_text, encoding='utf-8')
print(DATA_CFG.read_text(encoding='utf-8'))


In [ ]:
# 克隆你 GitHub 上的自定义仓库并本地安装
from pathlib import Path

REPO_URL = 'https://github.com/tuanziya666/yolov8s_kuangjing.git'
REPO_DIR = Path('/kaggle/working/yolov8s_kuangjing')

if REPO_DIR.exists():
    %cd /kaggle/working/yolov8s_kuangjing
    !git fetch origin
    !git reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /kaggle/working/yolov8s_kuangjing

!python -m pip uninstall -y ultralytics >/dev/null 2>&1 || true
!python -m pip install -q --no-deps -e .

import ultralytics, sys
print('python =', sys.executable)
print('ultralytics =', ultralytics.__file__)


In [ ]:
# 可选：清理旧结果
!rm -rf /kaggle/working/runs
!rm -rf /kaggle/working/evals
!rm -f /kaggle/working/yolov8s_gcblock_3cls_kaggle_results.zip


In [ ]:
# 修复 AMP 自检资源图
from pathlib import Path
import requests
from PIL import Image

assets_dir = Path('/kaggle/working/yolov8s_kuangjing/ultralytics/assets')
assets_dir.mkdir(parents=True, exist_ok=True)

def ensure_valid_image(path: Path, url: str) -> None:
    valid = False
    if path.exists():
        try:
            with Image.open(path) as im:
                im.verify()
            valid = True
        except Exception:
            path.unlink(missing_ok=True)

    if not valid:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        with Image.open(path) as im:
            im.verify()

ensure_valid_image(assets_dir / 'bus.jpg', 'https://ultralytics.com/images/bus.jpg')
ensure_valid_image(assets_dir / 'zidane.jpg', 'https://ultralytics.com/images/zidane.jpg')
print('assets fixed')


In [ ]:
# 公共训练参数
%cd /kaggle/working/yolov8s_kuangjing

import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_CFG = 'ultralytics/cfg/models/v8/yolov8s_gcblock.yaml'
EPOCHS = 300
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = '0'
PROJECT = '/kaggle/working/runs'
NAME = 'yolov8s_gcblock_3cls_kaggle'

print('MODEL_CFG =', MODEL_CFG)
print('DATA_CFG =', DATA_CFG)
print('EPOCHS =', EPOCHS)
print('BATCH =', BATCH)
print('DEVICE =', DEVICE)


In [ ]:
# 训练前清理显存
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('cuda ready =', torch.cuda.is_available())


In [ ]:
# 开始训练：Baseline + Neck-only GCBlock
%cd /kaggle/working/yolov8s_kuangjing

import os
from ultralytics import YOLO

for key in [
    'ULTRALYTICS_CLS_LOSS',
    'ULTRALYTICS_QFL_BETA',
    'ULTRALYTICS_IOU_LOSS',
    'ULTRALYTICS_INNER_IOU_RATIO',
    'ULTRALYTICS_WCIOU_AC_LAMBDA',
    'ULTRALYTICS_WCIOU_AC_GAMMA',
    'ULTRALYTICS_SA_BOX_ENABLE',
    'ULTRALYTICS_TAL_REG_ENABLE',
    'ULTRALYTICS_AUX_HEAD_ENABLE',
    'ULTRALYTICS_USE_DIFFICULTY_SAMPLER',
]:
    os.environ.pop(key, None)

os.environ['ULTRALYTICS_CLS_LOSS'] = 'bce'

model = YOLO(MODEL_CFG)
results = model.train(
    data=str(DATA_CFG),
    task='detect',
    project=PROJECT,
    name=NAME,
    device=DEVICE,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    workers=WORKERS,
    seed=SEED,
    deterministic=True,
    pretrained='yolov8s.pt',
    optimizer='SGD',
    cache=False,
    amp=True,
    save=True,
    val=True,
    plots=True,
    verbose=True,
    patience=100,
)
results


In [ ]:
# 检查训练输出
from pathlib import Path
import pandas as pd

run_dir = Path(PROJECT) / NAME
print('run_dir =', run_dir)
print('results.csv exists =', (run_dir / 'results.csv').exists())
print('best.pt exists =', (run_dir / 'weights' / 'best.pt').exists())
print('last.pt exists =', (run_dir / 'weights' / 'last.pt').exists())

if (run_dir / 'results.csv').exists():
    df = pd.read_csv(run_dir / 'results.csv')
    display(df.tail())


In [ ]:
# 用 best.pt 在 val / test 上评估
%cd /kaggle/working/yolov8s_kuangjing

from ultralytics import YOLO
from pathlib import Path

run_dir = Path(PROJECT) / NAME
best_pt = run_dir / 'weights' / 'best.pt'
eval_dir = Path('/kaggle/working/evals')
eval_dir.mkdir(parents=True, exist_ok=True)

model = YOLO(str(best_pt))

val_metrics = model.val(
    data=str(DATA_CFG),
    split='val',
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(eval_dir),
    name=f'{NAME}_val',
)

test_metrics = model.val(
    data=str(DATA_CFG),
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(eval_dir),
    name=f'{NAME}_test',
)

print('val done =', val_metrics is not None)
print('test done =', test_metrics is not None)


In [ ]:
# 打包结果，便于下载
%cd /kaggle/working
!zip -r yolov8s_gcblock_3cls_kaggle_results.zip runs evals coalmine3_kaggle.yaml
!ls -lh /kaggle/working/yolov8s_gcblock_3cls_kaggle_results.zip
